# Day C1 — 신경 데이터의 네 얼굴: spike, LFP, EEG/MEG, 칼슘 이미징

설명은 학습 사이트(또는 원본 레슨)에서 읽고, 코드는 이 노트북에서 위에서 아래로 차례대로 실행하세요. 앞 셀이 만든 변수와 파일을 뒤 셀이 씁니다.

원본 레슨: [lessons/advanced/C-neural-signals/day-C1.md](https://github.com/evansuhohyun-927/neurodegeneration-bioinformatics/blob/main/lessons/advanced/C-neural-signals/day-C1.md)

In [ ]:
# 레슨 코드가 쓰는 폴더를 미리 만든다 (Colab에서는 /content 아래에 생긴다)
import os
for d in ["data", "figures", "reports", "results", "../data", "../figures", "../reports"]:
    os.makedirs(d, exist_ok=True)

## 예시로 이해하기

### 예시 1 — 연속 기록을 epoch로 자르고 확인하기

In [ ]:
import numpy as np

rng = np.random.default_rng(0)
fs = 500                      # 1초에 500번 기록 (Hz)
n_ch = 4                      # 전극 4개
dur = 20                      # 20초 연속 기록
n = fs * dur
t = np.arange(n) / fs

# 연속 기록: 채널 × 시간. 모든 채널에 10 Hz 리듬 + 잡음
cont = 0.5 * np.sin(2 * np.pi * 10 * t) + rng.normal(0, 1, (n_ch, n))
print("연속 데이터 모양 (채널, 시간):", cont.shape)

# 자극이 2초마다 한 번씩 나왔다고 하자 (사건 시각, 초)
events = np.arange(1.0, 19.0, 2.0)
pre, post = 0.5, 1.0                       # 자극 전 0.5초 ~ 후 1.0초
i_pre, i_post = int(pre * fs), int(post * fs)
starts = (events * fs).astype(int)

# epoch 자르기: 채널 × 시간 × 시행
epochs = np.stack([cont[:, s - i_pre:s + i_post] for s in starts], axis=2)
ep_time = np.arange(-i_pre, i_post) / fs
print("epoch 모양 (채널, 시간, 시행):", epochs.shape)
print("epoch 시간축: %.3f초 ~ %.3f초" % (ep_time[0], ep_time[-1]))

# 확인 1: 3번째 시행, 2번 채널의 첫 값이 연속 데이터의 같은 위치와 같은가?
print("자른 값이 원본과 같은가:", np.allclose(epochs[1, :, 2], cont[1, starts[2] - i_pre:starts[2] + i_post]))

# 함정: reshape로 '대충' 자르면 순서가 섞인다
wrong = cont[:, :fs * 2 * 9].reshape(n_ch * 9, fs * 2)   # 행 = (채널, 조각)이 한 줄로 섞임
right = cont[:, :fs * 2 * 9].reshape(n_ch, 9, fs * 2)    # 채널은 그대로, 시간만 2초씩
print("wrong[1]이 '채널1'인가?", np.allclose(wrong[1], cont[1, :fs * 2]))
print("wrong[1]은 사실 채널0의 두 번째 2초:", np.allclose(wrong[1], cont[0, fs * 2:fs * 4]))
print("right[1, 0]은 채널1의 첫 2초:", np.allclose(right[1, 0], cont[1, :fs * 2]))

### 예시 2 — 그냥 솎아 내면 없던 리듬이 생긴다

In [ ]:
import numpy as np
from scipy import signal

fs = 1000                                   # 원래 1000 Hz로 기록
t = np.arange(0, 4, 1 / fs)
# 10 Hz 뇌 리듬 + 180 Hz 고주파 잡음(예: 근육 활동)
x = np.sin(2 * np.pi * 10 * t) + 0.8 * np.sin(2 * np.pi * 180 * t)

def amp_at(sig, rate, f):
    """진폭 스펙트럼에서 주파수 f(Hz)의 진폭"""
    amp = np.abs(np.fft.rfft(sig)) / len(sig) * 2
    hz = np.fft.rfftfreq(len(sig), 1 / rate)
    return round(float(amp[np.argmin(np.abs(hz - f))]), 3)

print("원본 1000 Hz  : 10 Hz=%.3f, 180 Hz=%.3f" % (amp_at(x, fs, 10), amp_at(x, fs, 180)))

# (1) 4개마다 하나씩 그냥 버리기 -> 250 Hz, Nyquist = 125 Hz
naive = x[::4]
print("그냥 솎아 냄  : 10 Hz=%.3f, 70 Hz=%.3f  <- 없던 70 Hz가 생겼다" % (amp_at(naive, 250, 10), amp_at(naive, 250, 70)))

# (2) 저역 통과 필터를 먼저 건 뒤 낮추기 (decimate가 둘 다 해 준다)
safe = signal.decimate(x, 4, ftype="fir", zero_phase=True)
print("decimate 사용 : 10 Hz=%.3f, 70 Hz=%.3f" % (amp_at(safe, 250, 10), amp_at(safe, 250, 70)))

### 예시 3 — 칼슘 형광을 ΔF/F로 바꾸고 사건 찾기

In [ ]:
import numpy as np
from scipy import ndimage

rng = np.random.default_rng(1)
fs = 30                                  # 칼슘 영상: 1초에 30장(frame)
t = np.arange(0, 60, 1 / fs)             # 60초
n = len(t)

# 1) 숨은 '진짜' 스파이크: 60초 동안 몇 번
spike_times = np.array([5.0, 5.3, 14.0, 27.5, 28.0, 28.4, 41.0, 52.2])
spikes = np.zeros(n)
spikes[np.round(spike_times * fs).astype(int)] = 1

# 2) 스파이크 하나가 남기는 칼슘 형광 모양: 빨리 오르고 천천히 감소(약 0.5초)
kt = np.arange(0, 3, 1 / fs)
kernel = (1 - np.exp(-kt / 0.05)) * np.exp(-kt / 0.5)
calcium = np.convolve(spikes, kernel)[:n]

# 3) 측정되는 형광 F = 기저 밝기(서서히 어두워짐: 광표백) + 신호 + 잡음
baseline = 100 * np.exp(-t / 300)
F = baseline * (1 + 0.6 * calcium) + rng.normal(0, 1.5, n)

# 4) ΔF/F: 기저 F0를 '이동 하위 백분위수'로 추정한 뒤 (F - F0) / F0
F0 = ndimage.percentile_filter(F, percentile=10, size=int(20 * fs))
dff = (F - F0) / F0

# 5) 사건 검출: 중앙값을 0으로 맞추고, 잡음 SD(중앙값 절대편차로 추정)의 4배를 넘는 '시작' 지점
dff = dff - np.median(dff)
noise_sd = 1.4826 * np.median(np.abs(dff))
above = dff > 4 * noise_sd
onsets = np.flatnonzero(above[1:] & ~above[:-1]) + 1
print("잡음 SD 추정: %.3f" % noise_sd)
print("검출된 사건 시작(초):", np.round(onsets / fs, 2))
print("실제 스파이크(초):    ", spike_times)
print("F의 처음/끝 평균 밝기: %.1f -> %.1f (광표백)" % (F[:fs].mean(), F[-fs:].mean()))
print("ΔF/F의 처음/끝 중앙값: %.3f -> %.3f" % (np.median(dff[:10 * fs]), np.median(dff[-10 * fs:])))